# Inventory Replenishment & Stockout Risk Dashboard  
## Data Cleaning and KPI Preparation

This notebook prepares the raw retail inventory dataset for SQL analysis and dashboard development. The goal is to clean the data, standardize the columns, check data quality, and create inventory-related KPI fields that can support business questions around sales performance, forecast accuracy, stockout risk, and overstock risk.

This step is important because the dashboard should not be built directly from messy raw data. Before creating SQL queries or Tableau visuals, I need to make sure the dataset is consistent, readable, and useful for supply chain analysis.

## 1. Import Python Libraries

In this step, I import the main Python libraries needed for the cleaning process. I use `pandas` for data loading, cleaning, and transformation. I use `numpy` for handling numerical calculations, especially when creating KPI fields and dealing with missing or infinite values.

These libraries help turn the raw CSV file into a cleaner dataset that can later be used in SQL and Tableau.

In [1]:
import pandas as pd 
import numpy as np 

## 2. Load the Raw Dataset

Here, I load the raw retail inventory CSV file into a pandas DataFrame. This is the starting point of the project. The raw dataset contains daily inventory, sales, demand forecast, pricing, store, product, category, and regional information.

Loading the data into Python allows me to inspect its structure and prepare it for analysis.

In [2]:
df = pd.read_csv("../data/raw/retail_inventory.csv")

df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


## 3. Inspect the Dataset Structure

Before cleaning the data, I first check the basic structure of the dataset. This includes the number of rows and columns, column names, data types, missing values, and duplicate records.

This step helps me understand whether the dataset is complete, whether any fields need type conversion, and whether there are obvious data quality issues that need to be fixed before analysis.

In [3]:
# Basic dataset inspection
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Shape: (73100, 15)

Columns:
Index(['Date', 'Store ID', 'Product ID', 'Category', 'Region',
       'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast',
       'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion',
       'Competitor Pricing', 'Seasonality'],
      dtype='str')

Data types:
Date                      str
Store ID                  str
Product ID                str
Category                  str
Region                    str
Inventory Level         int64
Units Sold              int64
Units Ordered           int64
Demand Forecast       float64
Price                 float64
Discount                int64
Weather Condition         str
Holiday/Promotion       int64
Competitor Pricing    float64
Seasonality               str
dtype: object

Missing values:
Date                  0
Store ID              0
Product ID            0
Category              0
Region                0
Inventory Level       0
Units Sold            0
Units Ordered         0
Demand Forec

### Initial Data Quality Notes

The initial inspection shows that the dataset is large enough for meaningful analysis and does not have major missing-value or duplicate-record issues. This is useful because it means the main cleaning work is not about fixing broken data, but about preparing the dataset for business analysis.

The most important next step is to standardize the column names so they are easier to use in Python, SQL, and Tableau.

## 4. Standardize Column Names

The original column names contain capital letters and spaces. While this is readable for humans, it can make analysis harder later, especially in SQL where spaces in column names can become annoying.

In this step, I standardize all column names by converting them to lowercase, removing extra spaces, and replacing spaces with underscores. This makes the dataset easier to query and reduces the chance of naming errors later in the project.

In [4]:
# Standardize column names for SQL/Python friendliness
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("/", "_")
)

df.columns

Index(['date', 'store_id', 'product_id', 'category', 'region',
       'inventory_level', 'units_sold', 'units_ordered', 'demand_forecast',
       'price', 'discount', 'weather_condition', 'holiday_promotion',
       'competitor_pricing', 'seasonality'],
      dtype='str')

## 5. Convert the Date Column

The `date` column needs converting from text into a proper datatime format. This is importatn because time-based analysis is a major part of the dashboard later on.

Once the date field is converted correctly and properly, I can create monthly trends, filter by time period, and analyze how sales and inventory behavior change over time.

In [5]:
df["date"] = pd.to_datetime(df["date"])

print(df["date"].min())
print(df["date"].max())

2022-01-01 00:00:00
2024-01-01 00:00:00


### Date Range Interpretation

The dataset covers inventory and sales activity from the beginning of 2022 to the beginning of 2024. This gives enough time coverage to look at monthly patterns and compare performance over multiple periods.

For the final dashboard, I need to be careful with incomplete months. If the final month only has partial data, it may create a misleading drop in the monthly trend chart.

## 6. Create Sales Revenue

Sales revenue is calculated by multiplying the number of units sold by the product price. This gives a basic measure of business performance at the product, store, category, and time-period level.

This field is important because many dashboard views need to show which products, stores, or categories generate the most revenue.

In [6]:
df["sales_revenue"] = df["units_sold"] * df["price"]

## 7. Create Inventory Value

Inventory value is calculated by multiplying the current inventory level by the product price. This gives an estimate of how much value is tied up in inventory.

This is useful from a supply chain and finance perspective because excess inventory can create working-capital issues. A product may generate revenue, but if it also holds too much inventory value, it may still require review.

In [11]:
df["inventory_value"] = df["inventory_level"] * df["price"]

## 8. Create Forecast Error Metrics

Forecast error compares actual units sold against forecasted demand. This helps measure how accurate demand planning is.

I create both forecast error and absolute forecast error. The regular forecast error shows whether actual sales were above or below forecast, while absolute forecast error focuses on the size of the mistake regardless of direction.

In [12]:
df["forecast_error"] = df["units_sold"] - df["demand_forecast"]

df["abs_forecast_error"] = df["forecast_error"].abs()

## 9. Create Forecast Accuracy

Forecast accuracy gives a simple estimate of how close the demand forecast was to actual sales. A higher value means the forecast was closer to actual demand.

This metric is important because poor forecast accuracy can lead to both stockouts and overstock. If demand is underestimated, the business may run out of inventory. If demand is overestimated, the business may carry too much inventory.

In [13]:
df["forecast_accuracy"] = 1 - (df["abs_forecast_error"] / df["demand_forecast"])

df["forecast_accuracy"] = df["forecast_accuracy"].replace([np.inf, -np.inf], np.nan)

df["forecast_accuracy"] = df["forecast_accuracy"].clip(lower=0, upper=1)

### Forecast Accuracy Interpretation

The forecast accuracy field is clipped between 0 and 1 so that the final metric stays within a realistic percentage range. A value close to 1 means high forecast accuracy, while a value closer to 0 means the forecast was less reliable.

This metric will later appear in the executive KPI cards and product performance table.

## 10. Create Inventory Coverage Ratio

Inventory coverage ratio compares inventory level against units sold. This gives a rough idea of how much inventory is available relative to demand.

This is not a perfect inventory planning metric because the dataset does not include full lead time, beginning inventory, or purchase-order records. However, it is still useful as a directional indicator of whether inventory may be too high or too low compared to sales activity.

In [14]:
df["inventory_coverage_ratio"] = df["inventory_level"] / df["units_sold"].replace(0, np.nan)

### Inventory Coverage Ratio Note

Some rows may have missing inventory coverage values because units sold can be zero. Dividing by zero would not be meaningful, so those values are left blank instead of forcing an inaccurate number.

This is acceptable because those missing values come from the logic of the calculation, not from a data loading error.

## 11. Create Stockout Risk Flag

Stockout risk is created by comparing inventory level against demand forecast. If inventory level is lower than forecasted demand, the row is flagged as a stockout risk.

This is useful because it helps identify products, categories, or stores where demand may exceed available inventory.

In [15]:
df["stockout_risk"] = np.where(df["inventory_level"] < df["demand_forecast"], 1, 0)

## 12. Create Overstock Risk Flag

Overstock risk is created by checking whether inventory level is more than twice the demand forecast. If inventory is much higher than expected demand, the row is flagged as an overstock risk.

This helps identify possible excess inventory. Overstock matters because it can tie up cash, increase holding costs, and reduce inventory efficiency.

In [17]:
df["overstock_risk"] = np.where(df["inventory_level"] > df["demand_forecast"] * 2, 1, 0)

## 13. Create Order-Sales Gap

The order-sales gap compares units ordered against units sold. This field is calculated as units ordered minus units sold.

A negative value means the product sold more units than were ordered during the selected period. A positive value means more units were ordered than sold. This metric should be interpreted carefully because the dataset does not include full purchase-order timing, supplier lead time, or beginning inventory.

In [18]:
df["net_inventory_change"] = df["units_ordered"] - df["units_sold"]

## 14. Create Time-Based Fields

In this step, I create extra date fields such as year, month, month name, quarter, and week. These fields make it easier to group and filter the data in SQL and Tableau.

Time-based fields are especially useful for the monthly sales trend chart, where the dashboard shows how revenue changes over time.

In [19]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.month_name()
df["quarter"] = df["date"].dt.quarter
df["week"] = df["date"].dt.isocalendar().week.astype(int)

## 15. Review the Cleaned Dataset

After creating the new KPI fields, I preview the cleaned dataset again. This helps confirm that the new columns were added correctly and that the data still looks reasonable.

This step is a final check before exporting the processed dataset for SQL analysis.

In [7]:
df.head()

,date,store_id,product_id,category,region,inventory_level,units_sold,units_ordered,demand_forecast,price,discount,weather_condition,holiday_promotion,competitor_pricing,seasonality,sales_revenue
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn,4254.50
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn,9451.50
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer,1819.35
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn,1995.92
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer,1030.96


## 16. Check Final Data Types and Summary Statistics

Before saving the cleaned data, I check the final data types and summary statistics. This helps confirm that numeric fields are stored correctly and that the new KPI columns have reasonable values.

This step is important because mistakes in data types or calculated fields can create incorrect SQL results and misleading dashboard visuals.

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 73100 entries, 0 to 73099
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   date                73100 non-null  datetime64[us]
 1   store_id            73100 non-null  str           
 2   product_id          73100 non-null  str           
 3   category            73100 non-null  str           
 4   region              73100 non-null  str           
 5   inventory_level     73100 non-null  int64         
 6   units_sold          73100 non-null  int64         
 7   units_ordered       73100 non-null  int64         
 8   demand_forecast     73100 non-null  float64       
 9   price               73100 non-null  float64       
 10  discount            73100 non-null  int64         
 11  weather_condition   73100 non-null  str           
 12  holiday_promotion   73100 non-null  int64         
 13  competitor_pricing  73100 non-null  float64       
 14  s

In [9]:
df.describe()

,date,inventory_level,units_sold,units_ordered,demand_forecast,price,discount,holiday_promotion,competitor_pricing,sales_revenue
count,73100,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000
mean,2023-01-01 00:00:00,274.469877,136.464870,110.004473,141.494720,55.135108,10.009508,0.497305,55.146077,7527.070929
min,2022-01-01 00:00:00,50.000000,0.000000,20.000000,-9.990000,10.000000,0.000000,0.000000,5.030000,0.000000
25%,2022-07-02 00:00:00,162.000000,49.000000,65.000000,53.670000,32.650000,5.000000,0.000000,32.680000,2024.632500
50%,2023-01-01 00:00:00,273.000000,107.000000,110.000000,113.015000,55.050000,10.000000,0.000000,55.010000,4956.090000
75%,2023-07-03 00:00:00,387.000000,203.000000,155.000000,208.052500,77.860000,15.000000,1.000000,77.820000,10618.605000
max,2024-01-01 00:00:00,500.000000,499.000000,200.000000,518.550000,100.000000,20.000000,1.000000,104.940000,47860.470000
std,NaN,129.949514,108.919406,52.277448,109.254076,26.021945,7.083746,0.499996,26.191408,7537.500441


## 17. Export the Cleaned Dataset

The final cleaned dataset is exported as a new CSV file. This file becomes the main processed dataset for the rest of the project.

After this step, the workflow moves from Python cleaning into SQL analysis, where I validate the data and create summary tables for dashboarding.

In [10]:
df.to_csv("../data/processed/cleaned_retail_inventory.csv", index=False)

## Notebook Summary

In this notebook, I cleaned the raw retail inventory dataset and created several business-focused KPI fields. The most important new fields include sales revenue, inventory value, forecast accuracy, stockout risk, overstock risk, inventory coverage ratio, and order-sales gap.

These fields help convert the dataset from a basic retail inventory file into a supply chain analysis dataset. The cleaned output can now support SQL validation, product-level analysis, category risk analysis, store performance review, and dashboard development.

The next step is to load the cleaned CSV into a SQLite database and use SQL to validate the data and create analysis outputs for the final dashboard.